# Trabajo Final - PEIA
Matías Souto - a2638

# Setup
Ejecutar la celda de abajo para instalar las dependencias en caso de corresponder

In [1]:
#!pip install numpy
#!pip install pandas
#!pip install scipy

---

# Contexto
Siguiendo con la historia de Don Francisco, con el tiempo y gracias a los análisis de Matías, el pequeño comerciante de
barrio cuenta hoy con 5 supermercados: ’Santa Ana’, ’La Floresta’, ’Los Cedros’, ’Palermo’ y ’Córdoba’.
También Matías ha avanzado en la Especialización en Inteligencia Artificial. Un día Don Francisco le plantea algunas
inquietudes adicionales:
1. Don Francisco quiere entender mejor la afluencia de clientes por mes del supermercado ’Santa Ana’.
2. Más aún, Don Francisco no sabe si puede estar seguro de que la afluencia de clientes son las mismas en todos los
supermercados o si hay alguno que se comporte mejor que los demás, y si alguna de las tiendas necesita más atención
porque está recibiendo menos clientes que las de las otras.

## Punto 1:Simulación de clientes diarios (Poisson)
Crear una simulación del número de clientes diarios que van a los almacenes de Don Francisco, usando
distribuciones Poisson, entre los años 2023, 2024 y 2025. En cada fecha, el parámetro $λt$ debe ser la suma de los efectos dados

Se simula $Y_{t,s} \sim \text{Poisson}(\lambda_{t,s})$ para cada fecha $t$ (2023-01-01 a 2025-12-31) y tienda $s$, donde $\lambda_{t,s}$ es la suma de 4 efectos (año, mes, día de semana, tienda), dados por las tablas del enunciado.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

SEED = 42

store_effect = pd.read_csv(f"./data/store.csv").rename(columns={"effect": "eff_store"})
year_effect = pd.read_csv(f"./data/year.csv").rename(columns={"effect": "eff_year"})
month_effect = pd.read_csv(f"./data/month.csv").rename(columns={"effect": "eff_month"})
day_effect = pd.read_csv(f"./data/day.csv").rename(columns={"effect": "eff_day"})

store_effect

,store,eff_store
0,Santa Ana,5000
1,La Floresta,2000
2,Los Cedros,3000
3,Palermo,1000
4,Córdoba,3000


In [3]:
fechas = pd.DataFrame({"date": pd.date_range("2023-01-01", "2025-12-31", freq="D")})
df = fechas.merge(store_effect, how="cross")

df["year"] = df["date"].dt.year
df["month_num"] = df["date"].dt.month
# Lunes=0 ... Domingo=6, igual que day.csv
df["day_num"] = df["date"].dt.weekday 

df = (
    df.merge(year_effect, on="year", how="left")
    .merge(month_effect, on="month_num", how="left")
    .merge(day_effect, on="day_num", how="left")
)

EFECTOS = ["eff_year", "eff_month", "eff_day", "eff_store"]
df["lambda"] = df[EFECTOS].sum(axis=1)

rng = np.random.default_rng(SEED)
df["clients"] = rng.poisson(df["lambda"])

df = df.sort_values(["date", "store"])[["date", "year", "month", "day", "store", *EFECTOS, "lambda", "clients"]].reset_index(drop=True)
print(df.shape)
df.head()

(5480, 11)


,date,year,month,day,store,eff_year,eff_month,eff_day,eff_store,lambda,clients
0,2023-01-01,2023,Enero,Domingo,Córdoba,1000,1000,1000,3000,6000,5899
1,2023-01-01,2023,Enero,Domingo,La Floresta,1000,1000,1000,2000,5000,5087
2,2023-01-01,2023,Enero,Domingo,Los Cedros,1000,1000,1000,3000,6000,5878
3,2023-01-01,2023,Enero,Domingo,Palermo,1000,1000,1000,1000,4000,4051
4,2023-01-01,2023,Enero,Domingo,Santa Ana,1000,1000,1000,5000,8000,8076


Se hacen verificaciones de sanidad de los datos y se guarda el consolidado en un csv dedicado

In [4]:
assert len(df) == 1096 * 5
assert df.isna().sum().sum() == 0
assert df["lambda"].min() == 4000 and df["lambda"].max() == 13500
print("Chequeos OK")

df.to_csv(f"./data/consolidated.csv", index=False)
df.groupby("store")["clients"].mean().sort_values(ascending=False)

Chequeos OK


store
Santa Ana      10716.547445
Los Cedros      8717.111314
Córdoba         8714.890511
La Floresta     7712.862226
Palermo         6718.302920
Name: clients, dtype: float64

## Punto 2:Intervalos de confianza para 'Santa Ana' por mes (95% y 99%)

Con base en los datos generados, determinen intervalos de confianza empíricos para el supermercado "Santa
Ana" en cada mes, para significancias del 95% y el 99%

**¿Por qué se puede usar la aproximación normal (z)?:** 

Porque una Poisson(λ) es la suma de λ variables Poisson, tal como se expresó en el punto 1, i.i.d.; así que, por el Teorema Central del Límite, para λ grande, Poisson(λ) ≈ Normal(λ, λ). En nuestros datos, los $λ$ diarios de Santa Ana van de ~8000 a ~11500, así que cada observación diaria es aproximadamente normal. Por eso se puede construir el intervalo con el estadístico z en vez de percentiles empíricos crudos:

$$\text{IC}_{1-\alpha} = \bar{x} \pm z_{1-\alpha/2}\cdot \sigma$$

donde $\sigma$ es el desvío estándar de los clientes diarios de cada mes (estimado con el desvío estándar muestral, dado que no se conoce el valor poblacional), que ya captura tanto el ruido Poisson como la variación entre años y días de la semana dentro del mes.

In [5]:
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]

Z95 = stats.norm.ppf(0.975)
Z99 = stats.norm.ppf(0.995)

santa_ana = df[df["store"] == "Santa Ana"]

filas = []
for mes in MESES:
    y = santa_ana.loc[santa_ana["month"] == mes, "clients"].to_numpy()
    media, s = y.mean(), y.std(ddof=1)
    filas.append({
        "month": mes, "n": y.size, "media": media, "desvio": s,
        "IC95": (media - Z95 * s, media + Z95 * s),
        "IC99": (media - Z99 * s, media + Z99 * s),
    })

ic_santa_ana = pd.DataFrame(filas)
ic_santa_ana

,month,n,media,desvio,IC95,IC99
0,Enero,93,9748.752688,1042.957309,"(7704.593924513094, 11792.91145183099)","(7062.27268850274, 12435.232687841344)"
1,Febrero,85,10215.847059,997.283841,"(8261.206647958108, 12170.48746968895)","(7647.014117033765, 12784.680000613293)"
2,Marzo,93,10673.526882,976.078997,"(8760.447201409832, 12586.60656203103)","(8159.313998505434, 13187.739764935428)"
3,Abril,90,10722.566667,1039.228950,"(8685.715353357902, 12759.417979975433)","(8045.690284663149, 13399.443048670186)"
4,Mayo,93,11252.612903,989.230630,"(9313.756495525127, 13191.469310926486)","(8704.523657850747, 13800.702148600867)"
5,Junio,90,11174.344444,1001.938180,"(9210.581697458214, 13138.107191430676)","(8593.522720777997, 13755.166168110893)"
6,Julio,93,11741.483871,1047.583810,"(9688.257332171856, 13794.71040976363)","(9043.086794709301, 14439.880947226184)"
7,Agosto,93,11220.397849,987.043578,"(9285.827985503256, 13154.967713421476)","(8677.942077491702, 13762.85362143303)"
8,Septiembre,90,11179.733333,1027.860621,"(9165.16353506803, 13194.303131598637)","(8532.139825826045, 13827.326840840622)"
9,Octubre,93,10734.989247,1022.658019,"(8730.616361598382, 12739.362133025272)","(8100.796754498677, 13369.181740124977)"


## Punto 3:ANOVA entre las 5 tiendas (α=0.05)

De igual manera, realicen pruebas ANOVA para determinar si los clientes esperados de todas las tiendas
son iguales o no, con significancia del 95

### Hipótesis

Debemos plantear la hipótesis nula como que el número promedio de clientes diarios es el mismo en las cinco tiendas, es decir que la tienda no tiene ningún efecto sobre la afluencia. La hipótesis alternativa debe ser que que al menos una tienda tiene un promedio distinto de las demás.

**Todas las tiendas tienen la misma media** -> $H_0: \mu_{SA}=\mu_{LF}=\mu_{LC}=\mu_{P}=\mu_{C}$

**Existe al menos una tienda cuya media es diferente a las otras** -> $H_1: \exists\, i \neq j \;/\; \mu_i \neq \mu_j$

### Chequeo de supuestos

El ANOVA es válido solo si se cumplen ciertas condiciones sobre los datos; antes de interpretar el resultado del test se verifica cada una:

- **Independencia**: las observaciones de una tienda no pueden estar relacionadas entre sí ni con las de otra tienda, porque si lo estuvieran el test subestimaría la variabilidad real y aumentaría el riesgo de rechazar $H_0$ sin fundamento. Se cumple por construcción: cada `clients` se muestreó de forma independiente (una Poisson por fila).
- **Normalidad**: el estadístico F del ANOVA se deriva asumiendo que, dentro de cada grupo, los datos siguen una distribución normal. Ya se justificó en el Punto 2: Poisson(λ) ≈ Normal(λ,λ) para λ grande (todos los λ acá son ≥4000).
- **Homocedasticidad** (varianzas iguales entre tiendas): el ANOVA compara la variabilidad entre grupos contra la variabilidad dentro de los grupos, y esa comparación solo es justa si los grupos tienen varianzas similares; si no, una tienda más "ruidosa" podría inflar el residuo y ocultar diferencias reales entre medias. Se verifica con el test de Bartlett.

In [6]:
TIENDAS = ["Santa Ana", "La Floresta", "Los Cedros", "Palermo", "Córdoba"]
grupos = [df.loc[df["store"] == t, "clients"].to_numpy(dtype=float) for t in TIENDAS]

bart_stat, bart_p = stats.bartlett(*grupos)
print(f"Bartlett: p-valor = {bart_p:.4f}")
print("Varianzas homogéneas" if bart_p > 0.05 else "Varianzas heterogéneas")

Bartlett: p-valor = 0.9999
Varianzas homogéneas


In [7]:
F, p_valor = stats.f_oneway(*grupos)

print(f"F = {F:.4f}   p-valor = {p_valor:.3e}")
print("Se rechaza H0: las medias no son todas iguales" if p_valor < 0.05 else "No se rechaza H0")

F = 1723.4975   p-valor = 0.000e+00
Se rechaza H0: las medias no son todas iguales


## Punto 4:Tienda con mayor vs menor promedio

Finalmente, identifiquen la tienda con mayor promedio y la tienda con menor promedio de clientes y realicen
una prueba de hipótesis para determinar si la diferencia entre ellas es distinta de cero o no. Verifiquen si las tiendas
identificadas corresponden a las tiendas con mayores y menores efectos

### Hipótesis

La hipótesis nula debería plantearse como que la tienda con mayor promedio observado y la de menor promedio observado tienen la misma media poblacional de clientes diarios, y que la diferencia vista en la muestra se debe solo al azar. La hipótesis alternativa afirma que esa diferencia es real y distinta de cero.

**La diferencia de medias es cero** -> $H_0: \mu_{max} - \mu_{min} = 0$
**La diferencia de medias es distinta de cero** -> $H_1: \mu_{max} - \mu_{min} \neq 0$

Se usa el estadístico z de dos muestras independientes (n=1096 por tienda), con el mismo argumento de normalidad ya justificado en el Punto 2 (Poisson(λ) ≈ Normal(λ,λ) para λ grande).

In [8]:
medias = df.groupby("store")["clients"].mean().sort_values(ascending=False)
t_max, t_min = medias.index[0], medias.index[-1]

y_max = df.loc[df["store"] == t_max, "clients"].to_numpy(dtype=float)
y_min = df.loc[df["store"] == t_min, "clients"].to_numpy(dtype=float)

dif = y_max.mean() - y_min.mean()
ee = np.sqrt(y_max.var(ddof=1) / y_max.size + y_min.var(ddof=1) / y_min.size)
z_obs = dif / ee
p_valor_z = 2 * stats.norm.sf(abs(z_obs))

print(f"Mayor promedio: {t_max} ({medias.iloc[0]:.1f})")
print(f"Menor promedio: {t_min} ({medias.iloc[-1]:.1f})")
print(f"\nDiferencia = {dif:.1f}   z = {z_obs:.2f}   p-valor = {p_valor_z:.3e}")
print("Se rechaza H0: la diferencia es distinta de cero" if p_valor_z < 0.05 else "No se rechaza H0")

efectos = store_effect.set_index("store")["eff_store"]
e_max, e_min = efectos.idxmax(), efectos.idxmin()
print(f"\nTienda de mayor efecto teórico: {e_max}  ->  coincide con observado: {'sí' if e_max == t_max else 'no'}")
print(f"Tienda de menor efecto teórico: {e_min}  ->  coincide con observado: {'sí' if e_min == t_min else 'no'}")

Mayor promedio: Santa Ana (10716.5)
Menor promedio: Palermo (6718.3)

Diferencia = 3998.2   z = 79.20   p-valor = 0.000e+00
Se rechaza H0: la diferencia es distinta de cero

Tienda de mayor efecto teórico: Santa Ana  ->  coincide con observado: sí
Tienda de menor efecto teórico: Palermo  ->  coincide con observado: sí
